In [1]:
import numpy as np
import tensorflow as tf
import sys

sys.path.insert(1, './seld-net')
import cls_data_generator
import parameter
import evaluation_metrics


In [2]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print(tf.config.experimental.list_physical_devices('GPU'))


Num GPUs Available:  0
[]


In [3]:
model = tf.keras.models.load_model('./models/drone_ov1_split1_regr0_3d0_2_model.keras')

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 16, 512,   │          0 │ -                 │
│ (InputLayer)        │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 512,   │      9,280 │ input_layer[0][0] │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 512,   │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 512,   │          0 │ batch_normalizat… │
│ (Activation)        │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 512,   │          0 │ activation[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64, 512,   │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 512,   │     36,928 │ dropout[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 512,   │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 512,   │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 512,   │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64, 512,   │          0 │ max_pooling2d_1[… │
│                     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 512,   │     36,928 │ dropout_1[0][0]   │
│                     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 512,   │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 512,   │          0 │ batch_normalizat… │
│ (Activation)        │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 64, 512,   │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64, 512,   │          0 │ max_pooling2d_2[… │
│                     │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute (Permute)   │ (None, 512, 64,   │          0 │ dropout_2[0][0] 

 Total params: 1,540,430 (5.88 MB)

 Trainable params: 513,348 (1.96 MB)

 Non-trainable params: 384 (1.50 KB)

 Optimizer params: 1,026,698 (3.92 MB)

In [4]:
params = parameter.get_params("9")

data_gen_test = cls_data_generator.DataGenerator(
        dataset=params['dataset'], ov=params['overlap'], split=params['split'], db=params['db'], nfft=params['nfft'],
        batch_size=params['batch_size'], seq_len=params['sequence_length'], classifier_mode=params['mode'],
        weakness=params['weakness'], datagen_mode='test', cnn3d=params['cnn_3d'], xyz_def_zero=params['xyz_def_zero'],
        azi_only=params['azi_only'], shuffle=False
    )

SET: 9
quick_test: False
azi_only: False
dataset: drone
overlap: 1
split: 1
db: 50
nfft: 512
sequence_length: 512
batch_size: 4
dropout_rate: 0.0
nb_cnn2d_filt: 64
pool_size: [8, 8, 2]
rnn_size: [128, 128]
fnn_size: [128]
loss_weights: [1.0, 50.0]
xyz_def_zero: True
nb_epochs: 1000
mode: regr
nb_cnn3d_filt: 32
cnn_3d: False
weakness: 0
patience: 100
Datagen_mode: test, nb_files: 60, nb_classes:1
nb_frames_file: 5166, feat_len: 256, nb_ch: 16, label_len:3

Dataset: drone, ov: 1, split: 1
batch_size: 4, seq_len: 512, shuffle: False
label_dir: ./dataset_generator/sounds/filtered_8_channel_microphone_signals\label_ov1_split1_nfft512_regr0
 feat_dir: ./dataset_generator/sounds/filtered_8_channel_microphone_signals\spec_ov1_split1_50db_nfft512_norm



In [5]:
def remove_label():
    temp = next(data_gen_test.generate())
    out1 = temp[0][0:1, :, :, :]
    yield (out1,)

In [10]:
for i in range(3):
    pred = model.predict(
        remove_label(),
        # steps=2 if params['quick_test'] else data_gen_test.get_total_batches_in_data(),
        steps=1,
        verbose=1
    )

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step


In [ ]:
pred[0].shape

(1, 512, 1)

In [ ]:
sed_pred = evaluation_metrics.reshape_3Dto2D(pred[0]) > 0.5
doa_pred = evaluation_metrics.reshape_3Dto2D(pred[1])

In [ ]:
sed_gt = np.zeros_like(sed_pred)
doa_gt = np.zeros_like(doa_pred)

i = 1
for feat, label in data_gen_test.generate():
    sed_gt[(i-1) * params['sequence_length'] * params['batch_size'] : i * params['sequence_length'] * params['batch_size'], : ] = evaluation_metrics.reshape_3Dto2D(label[0])
    doa_gt[(i-1) * params['sequence_length'] * params['batch_size'] : i * params['sequence_length'] * params['batch_size'], : ] = evaluation_metrics.reshape_3Dto2D(label[1])
    i += 1
    if i == data_gen_test.get_total_batches_in_data():
        break

In [ ]:
doa_loss, conf_mat = evaluation_metrics.compute_doa_scores_regr_xyz(doa_pred, doa_gt, sed_pred, sed_gt)
sed_loss = evaluation_metrics.compute_sed_scores(sed_pred, sed_gt, data_gen_test.nb_frames_1s())

In [46]:
import pyaudio
import wave

FORMAT = pyaudio.paInt16
CHANNELS = 2
RATE = 44100
CHUNK = 1024
RECORD_SECONDS = 5
WAVE_OUTPUT_FILENAME = "./file.wav"

audio = pyaudio.PyAudio()

# Open input stream (microphone)
input_stream = audio.open(format=FORMAT,
                          channels=CHANNELS,
                          rate=RATE,
                          input=True,
                          frames_per_buffer=CHUNK)

# Open output stream (speaker)
output_stream = audio.open(format=FORMAT,
                           channels=CHANNELS,
                           rate=RATE,
                           output=True,
                           frames_per_buffer=CHUNK)

print("Recording with real-time echo...")

frames = []

for _ in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = input_stream.read(CHUNK)
    frames.append(data)
    output_stream.write(data)  # Echo audio in real time

print("Done recording.")

# Stop and close streams
input_stream.stop_stream()
input_stream.close()
output_stream.stop_stream()
output_stream.close()
audio.terminate()
 
# waveFile = wave.open(WAVE_OUTPUT_FILENAME, 'wb')
# waveFile.setnchannels(CHANNELS)
# waveFile.setsampwidth(audio.get_sample_size(FORMAT))
# waveFile.setframerate(RATE)
# waveFile.writeframes(b''.join(frames))
# waveFile.close()

ALSA lib pcm_dsnoop.c:567:(snd_pcm_dsnoop_open) unable to open slave
ALSA lib pcm_dmix.c:1000:(snd_pcm_dmix_open) unable to open slave
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
Cannot connect to server socket err = No such file or directory
Cannot connect to server request channel
jack server is not running or cannot be started
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
Cannot connect to server socket err = No such file or directory
Cannot connect to server request channel
jack server is not running or cannot be started
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
ALSA lib pcm

Recording with real-time echo...
Done recording.
